# Contrastive learning (SimCLR, CLIP)

_Modern Deep Learning & AI_

**Pull two views of the same thing together, push everything else apart. No labels needed.**

Labels are expensive. Self-supervised learning learns useful features from raw data with no human labels.

     Contrastive learning is one way. Take an image, make two altered copies (crop, recolour). These are a positive pair: they should land close together in feature space.

---

This notebook is a practice scaffold for the **Contrastive learning (SimCLR, CLIP)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F

def nt_xent(z1, z2, tau=0.1):
    # z1, z2: (B, d) embeddings of the two views of the same B inputs
    B = z1.size(0)
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)  # (2B, d) unit vectors
    sim = (z @ z.t()) / tau                              # cosine sim matrix (2B, 2B)
    sim.fill_diagonal_(float('-inf'))                    # no self-comparison
    # positive of row i is its partner B positions away (and vice versa)
    targets = torch.cat([torch.arange(B, 2 * B),
                         torch.arange(0, B)])
    return F.cross_entropy(sim, targets)                 # InfoNCE = softmax CE

# synthetic batch: 4 items, 8-dim embeddings, two views each
z1 = torch.randn(4, 8)
z2 = z1 + 0.05 * torch.randn(4, 8)   # view 2 is a slightly perturbed view 1
loss = nt_xent(z1, z2, tau=0.1)
print(float(loss))

## Visualize the data & results

_On real handwritten-digit images, does the cosine-similarity matrix light up where two samples of the same digit meet?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()                # real 8x8 handwritten digit images
X, y = digits.data, digits.target

# Two real samples of each digit class 0,1,2,3 = the positive pairs.
rows = []
names = []
for tag, k in [('a', 0), ('b', 1)]:
    for d in [0, 1, 2, 3]:
        rows.append(X[y == d][k]); names.append(f'd{d}_{tag}')
Z = np.array(rows, dtype=float)
Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)   # L2-normalize
S = Z @ Z.T                                                  # cosine similarity

plt.figure(figsize=(6, 5))
plt.imshow(S, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='cosine similarity')
plt.xticks(range(8), names, rotation=45); plt.yticks(range(8), names)
plt.title('Cosine similarity of real digit embeddings')
for i in range(8):
    for j in range(8):
        plt.text(j, i, format(S[i, j], '.2f'), ha='center', va='center', fontsize=7)
plt.tight_layout(); plt.show()